# Notebook 03 — Mise à Jour Quotidienne (Pipeline Expert)

Applique les modèles **Émotions** (MLP) + **Fake News Expert** (bilingue) sur les nouveaux posts Bluesky.

**Améliorations par rapport à la version précédente :**
- Utilisation du modèle expert (sans biais Reuters)
- Détection de langue automatique (FR/EN)
- Features linguistiques (12 signaux)
- Monitoring CodeCarbon intégré

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
from pymongo import MongoClient, UpdateOne
from codecarbon import EmissionsTracker

from pipeline.expert_detector import ExpertFakeNewsDetector

print('Imports OK')

In [ ]:
print('='*60)
print('MISE À JOUR QUOTIDIENNE — Pipeline Expert')
print('='*60)

# --- 1. CHARGEMENT DES MODÈLES ---

# A. Modèle Émotions (MLP PyTorch bilingue)
print('\nChargement du modèle ÉMOTIONS (MLP PyTorch bilingue)...')

VOCAB_SIZE = 25000
MAX_LENGTH = 60

class EmotionMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 48)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(48, 24)
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(24, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = torch.relu(self.fc1(x))
        x = self.drop1(x)
        x = torch.relu(self.fc2(x))
        x = self.drop2(x)
        return self.fc3(x)

with open('../models/emotion_vocab_bilingual.pickle', 'rb') as f:
    emotion_vocab = pickle.load(f)
with open('../models/emotion_label_encoder_bilingual.pickle', 'rb') as f:
    emotion_le = pickle.load(f)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_emotion = EmotionMLP(VOCAB_SIZE, 64, len(emotion_le.classes_)).to(device)
model_emotion.load_state_dict(torch.load('../models/emotion_bilingual.pt', map_location=device, weights_only=True))
model_emotion.eval()

def predict_emotion_proba(texts):
    """Retourne array (n, 7) de probabilites d'emotion."""
    oov_idx = emotion_vocab['<OOV>']
    sequences = []
    for text in texts:
        tokens = str(text).lower().split()
        seq = [emotion_vocab.get(t, oov_idx) for t in tokens[:MAX_LENGTH]]
        seq = seq + [0] * (MAX_LENGTH - len(seq))
        sequences.append(seq)
    X = torch.tensor(sequences, dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model_emotion(X)
        return torch.softmax(logits, dim=1).cpu().numpy()

print(f'  Classes : {list(emotion_le.classes_)}')

# B. Modèle Fake News Expert (bilingue)
print('Chargement du modèle FAKE NEWS Expert (bilingue)...')
detector = ExpertFakeNewsDetector(model_dir='../models')
detector.load(suffix='expert')
print(f"  Modèle : {detector.training_metrics.get('model_type', '?').upper()}")
print(f"  F1 (CV) : {detector.training_metrics.get('cv_f1_mean', '?')}")

print('\nTous les modèles sont chargés.')

In [ ]:
# --- 2. RÉCUPÉRATION DES DONNÉES (incrémental : uniquement les posts non analysés) ---
client = MongoClient('mongodb://mongodb:27017/')
db = client['thumalien_db']
collection = db['raw_posts']

# Filtre incrémental : ne récupère que les posts sans version de modèle
cursor = collection.find(
    {'ai_model_version': {'$exists': False}},
    {'_id': 0, 'uri': 1, 'text': 1}
)
df = pd.DataFrame(list(cursor))
print(f'Nouveaux posts à analyser : {len(df)}')

total_posts = collection.count_documents({})
already_done = collection.count_documents({'ai_model_version': {'$exists': True}})
print(f'  (total en base : {total_posts}, déjà analysés : {already_done})')

if df.empty:
    print('Aucun nouveau post à analyser. Pipeline à jour.')
else:
    # --- 3. PRÉDICTION ÉMOTIONS ---
    print('\nAnalyse 1/2 : Émotions...')
    tracker = EmissionsTracker(project_name='Thumalien_Daily_Emotions', output_dir='../')
    tracker.start()

    pred_emotions = predict_emotion_proba(df['text'].astype(str).values)
    df['ai_emotion'] = emotion_le.inverse_transform(np.argmax(pred_emotions, axis=1))

    em_emotions = tracker.stop()
    print(f'  CO2 émis (émotions) : {em_emotions:.10f} kg')

    # --- 4. PRÉDICTION FAKE NEWS (Expert bilingue) ---
    print('Analyse 2/2 : Fake News Expert (bilingue)...')
    tracker2 = EmissionsTracker(project_name='Thumalien_Daily_FakeNews', output_dir='../')
    tracker2.start()

    predictions = detector.predict(df['text'])
    df['ai_score_credibility'] = predictions['ai_score_credibility'].values
    df['prediction_label'] = predictions['prediction_label'].values
    df['ai_language'] = predictions['language'].values
    df['ai_analysis_log'] = predictions['ai_analysis_log'].values

    em_fake = tracker2.stop()
    print(f'  CO2 émis (fake news) : {em_fake:.10f} kg')

    # --- 5. SAUVEGARDE MONGODB ---
    print(f'\nSauvegarde de {len(df)} analyses...')
    ops = []
    for _, row in df.iterrows():
        ops.append(
            UpdateOne(
                {'uri': row['uri']},
                {'$set': {
                    'ai_emotion': row['ai_emotion'],
                    'ai_score_credibility': float(row['ai_score_credibility']),
                    'prediction_label': int(row['prediction_label']),
                    'ai_language': row['ai_language'],
                    'ai_analysis_log': row['ai_analysis_log'],
                    'ai_model_version': 'expert_v2',
                }}
            )
        )

    if ops:
        collection.bulk_write(ops)

    # --- RÉSUMÉ ---
    print()
    print('='*60)
    print('RÉSUMÉ')
    print('='*60)
    print(f"  Nouveaux posts analysés : {len(df)}")
    print(f"  Total en base           : {total_posts}")
    print(f"  Langues détectées : {df['ai_language'].value_counts().to_dict()}")
    print(f"  Émotions          : {df['ai_emotion'].value_counts().to_dict()}")
    print(f"  Fake News         : {df['prediction_label'].value_counts().rename({0:'Fiable', 1:'Suspect'}).to_dict()}")
    print(f"  Crédibilité moy.  : {df['ai_score_credibility'].mean():.2%}")
    print(f"  CO2 total         : {(em_emotions + em_fake):.10f} kg")
    print()
    print('Dashboard prêt ! Rafraîchissez la page Streamlit.')